# Forget-MI **SUA THEO BAI BAO** (Kaggle, 30 epoch)

Chay `training/forgetmi_paper_fixed.py` - sua cac loi cua code goc **theo dung cong thuc bai bao**
(Hardan et al., MICCAI 2025), KHONG them y tuong moi.

## PHAT HIEN QUAN TRONG
Ban tai lap TRUNG THANH (`forgetmi_partial.py`) ra **Df-AUC 0.561 / Df-F1 0.156 / MIA 1.000**.
Doi chieu Bang 2 bai bao (3%), con so nay khop nhat voi bien the **"No Noise"**
(0.534 / 0.116 / 1.000) - **bien the bai bao noi cho ket qua TE NHAT**, KHONG phai Forget-MI that.

Forget-MI that @3% (Unimodal) = **Df-AUC 0.735 / Df-F1 0.393 / MIA 0.571**.

## Cac loi da sua (loi code goc -> bai bao -> cach sua)
| # | Loi code goc | Bai bao noi gi | Sua |
|---|---|---|---|
| 1 | Gate tao MOI ngau nhien MOI batch; F_ul va F_og dung 2 gate KHAC nhau => `L_MU`/`L_MR` la nhieu | "joint embeddings using a multimodal adaptation gate" (1 khoi kien truc co dinh) | MOT gate duy nhat, tao 1 lan, DONG BANG, dung chung |
| 2 | `og_frgt_joint` dung nham gate + truyen img 2 lan | Eq(2) chi can F_og tren ban NHIEU | bo (code chet) |
| 3 | `optimizer.step()` 1 lan/epoch; epoch 0 khong update => 29 update | chuan: cap nhat theo batch | step MOI batch, train tu epoch 0 (~390 step) |
| 4 | hinge `min(L_UR, margin)` | Eq(3),(4) la khoang cach THUAN | bo hinge |
| 5 | `model_og.train()` (BN batch-stats) | Fig.2: F_og **Frozen** | `.eval()` + `requires_grad=False` |
| 6 | `use_noise=True` lat dau -> cuc TIEU hoa khoang cach | Eq(1),(2) LUON la `-Dist` | luon `-Dist` |
| 7 | trong so khong theo preset | Eq(5) `sum(w)=1`; best@3% = **Unimodal** | preset equal/unimodal/multimodal/retention |

Eval tren `D_t_final` (75% test) bang CUNG pipeline voi P3-P6/main => so sanh duoc.
F_ul tinh chinh **toan bo** (paper khong dung LoRA). **Uoc tinh ~1.5-2.5h/preset.**


In [ ]:
# Cell 1: setup repo + deps
import os, subprocess
WORK_DIR = '/kaggle/working'
REPO_DIR = f'{WORK_DIR}/Forget-MI-LoKU'
REPO_URL = 'https://github.com/nhnhu146/Forget-MI-LoKU.git'
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
os.chdir(REPO_DIR)
assert os.path.exists('training/forgetmi_paper_fixed.py'), \
    'Chua thay training/forgetmi_paper_fixed.py - commit+push roi re-import notebook.'
subprocess.run(['pip', 'install', '-q', 'pydicom', 'scikit-image', 'scikit-learn',
                'pyyaml', 'wandb', 'seaborn==0.13.2'], check=True)
subprocess.run(['pip', 'install', '-q', 'transformers==4.38.0', 'peft==0.10.0',
                'accelerate==0.27.0'], check=True)
import torch
assert torch.cuda.is_available(), 'Bat GPU trong Kaggle Settings truoc khi chay.'
print('Commit:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())
print('GPU   :', torch.cuda.get_device_name(0))


In [ ]:
# Cell 2: CO + path discovery
import glob, os

FORGET_PCT = 3        # 3 | 6 | 10
SEED       = 42
LR         = 1e-5     # paper dung 1e-4 hoac 1e-5

# Preset trong so muon chay. 'auto' = preset TOT NHAT theo bai bao cho ti le nay
#   (3% -> unimodal, 6% -> multimodal, 10% -> retention).
# Muon tai lap ca Bang 2 thi dat: ['unimodal','multimodal','equal','retention'] (~4x thoi gian).
PRESETS_TO_RUN = ['auto']

assert FORGET_PCT in (3, 6, 10)

def find_dataset(*slugs):
    for slug in slugs:
        direct = f'/kaggle/input/{slug}'
        if os.path.isdir(direct):
            return direct
        hits = glob.glob(f'/kaggle/input/datasets/*/{slug}')
        if hits:
            return sorted(hits)[0]
    return None

def first_existing(root, relatives):
    for rel in relatives:
        p = os.path.join(root, rel)
        if os.path.exists(p):
            return p
    return None

DATA_ROOT = find_dataset('forget-mi-data')
MODELS_ROOT = find_dataset('forget-mi-models-full', 'forget-mi-models-v2', 'forget-mi-models')
assert DATA_ROOT and MODELS_ROOT, 'Add forget-mi-data va forget-mi-models(-full) vao Kaggle.'

base_hits = glob.glob(os.path.join(MODELS_ROOT, '**', 'training_original_model', 'pytorch_model.bin'), recursive=True)
gold_hits = glob.glob(os.path.join(MODELS_ROOT, '**', f'model_retrained_{FORGET_PCT}per', '**', 'pytorch_model.bin'), recursive=True)
BASE_MODEL = os.path.dirname(sorted(base_hits, key=len)[0]) if base_hits else None
GOLD_MODEL = os.path.dirname(sorted(gold_hits, key=len)[0]) if gold_hits else BASE_MODEL
TEXT_DIR = first_existing(DATA_ROOT, ['data/metadata', 'metadata'])
IMG_DIR  = first_existing(DATA_ROOT, ['data/img_data', 'img_data'])
FORGET_CSV  = f'./data_splits/forget_set_{FORGET_PCT}per.csv'
RESULTS_CSV = '/kaggle/working/results_advanced.csv'
HISTORY_CSV = f'/kaggle/working/perepoch_paperfix_{FORGET_PCT}per_s{SEED}.csv'
OUTPUT_DIR  = f'/kaggle/working/paperfix_output/{FORGET_PCT}per_s{SEED}'

for name, path in {'base model': BASE_MODEL, 'text metadata': TEXT_DIR,
                   'images': IMG_DIR, 'forget csv': FORGET_CSV}.items():
    assert path and os.path.exists(path), f'Missing {name}: {path}'

COMMON_OVR = {
    'forget_set_path': FORGET_CSV,
    'base_model_path': BASE_MODEL,
    'bert_pretrained_dir': BASE_MODEL,
    'retrained_model_path': GOLD_MODEL,
    'text_data_dir': TEXT_DIR,
    'img_data_dir': IMG_DIR,
    'output_dir': OUTPUT_DIR,
    'results_csv_path': RESULTS_CSV,
    'history_csv_path': HISTORY_CSV,
    'learning_rate': LR,
    'unlearn_batch_size': 8,   # train FULL 113M fp32 tren anh 2048^2 -> batch 16 OOM tren T4 14.5GB
    'eval_batch_size': 8,
}
print('FORGET_PCT :', FORGET_PCT, '| SEED', SEED, '| LR', LR)
print('BASE_MODEL :', BASE_MODEL)
print('PRESETS    :', PRESETS_TO_RUN)


In [ ]:
# Cell 3: CHAY Forget-MI SUA-THEO-BAI-BAO (chiu loi)
import os, subprocess, time
RUN_LOG = []

def run_preset(preset):
    rid = f'forgetmi_paperfix_{preset}_{FORGET_PCT}per_s{SEED}'
    ovr = dict(COMMON_OVR); ovr['weight_preset'] = preset; ovr['id'] = rid
    arg = ','.join(f'{k}={v}' for k, v in ovr.items())
    env = {**os.environ, 'PYTHONPATH': '.', 'WANDB_MODE': 'disabled',
           'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True'}
    cmd = ['python', 'training/forgetmi_paper_fixed.py', '--config', 'config_advanced_kaggle.yaml',
           '--seed', str(SEED), '--override', arg]
    print(chr(10) + '=' * 70 + f'{chr(10)}RUN {rid}{chr(10)}' + '=' * 70)
    t0 = time.time()
    try:
        subprocess.run(cmd, env=env, check=True)
        RUN_LOG.append((rid, 'OK', round((time.time() - t0) / 3600, 2)))
    except subprocess.CalledProcessError as e:
        print(f'FAILED {rid} rc={e.returncode} - chay tiep preset sau')
        RUN_LOG.append((rid, f'FAIL rc={e.returncode}', round((time.time() - t0) / 3600, 2)))

for p in PRESETS_TO_RUN:
    run_preset(p)

print(chr(10) + '=' * 70 + f'{chr(10)}TONG KET RUN:')
for rid, st, h in RUN_LOG:
    print(f'  [{st:>10}] {rid}  ({h}h)')


In [ ]:
# Cell 4: DOI CHIEU voi Bang 1 & 2 cua BAI BAO
import os, pandas as pd

# (MIA, Df-AUC, Df-F1, Dt-AUC, Dt-F1) - Bang 2 bai bao
PAPER = {
 3:  {'Original':(1.000,0.999,0.965,0.677,0.388), 'Retrain':(0.000,0.566,0.310,0.626,0.362),
      'No Noise':(1.000,0.534,0.116,0.508,0.154), 'Equal':(0.571,0.764,0.385,0.631,0.253),
      'Multimodal':(0.714,0.774,0.384,0.634,0.247), 'Unimodal*':(0.571,0.735,0.393,0.625,0.250),
      'Retention':(0.714,0.646,0.306,0.598,0.245)},
 6:  {'Original':(1.000,0.999,0.972,0.677,0.388), 'Retrain':(0.769,0.675,0.395,0.702,0.427),
      'No Noise':(1.000,0.511,0.110,0.521,0.090), 'Equal':(0.846,0.802,0.428,0.637,0.277),
      'Multimodal*':(0.615,0.654,0.328,0.599,0.270), 'Unimodal':(0.615,0.687,0.357,0.610,0.275),
      'Retention':(0.615,0.672,0.339,0.594,0.267)},
 10: {'Original':(1.000,0.999,0.970,0.677,0.388), 'Retrain':(0.190,0.588,0.342,0.629,0.382),
      'No Noise':(1.000,0.513,0.104,0.505,0.085), 'Equal':(0.952,0.796,0.369,0.617,0.240),
      'Multimodal':(0.905,0.740,0.344,0.586,0.231), 'Unimodal':(0.905,0.764,0.363,0.597,0.236),
      'Retention*':(0.810,0.656,0.313,0.565,0.252)},
}
print(f'===== BAI BAO (Bang 2, {FORGET_PCT}%) — * = best theo paper =====')
print(f"{'Bien the':<14}{'MIA':>8}{'Df-AUC':>9}{'Df-F1':>8}{'Dt-AUC':>9}{'Dt-F1':>8}")
for k, v in PAPER[FORGET_PCT].items():
    print(f'{k:<14}{v[0]:>8.3f}{v[1]:>9.3f}{v[2]:>8.3f}{v[3]:>9.3f}{v[4]:>8.3f}')

print(chr(10) + '===== TAI LAP CUA TA (ban SUA theo bai bao) =====')
if os.path.exists(RESULTS_CSV):
    df = pd.read_csv(RESULTS_CSV)
    df = df[df['method'] == 'forgetmi_paper'] if 'method' in df.columns else df
    if len(df):
        print(f"{'run':<34}{'MIA_p':>8}{'Df-AUC':>9}{'Df-F1':>8}{'Dt-AUC':>9}{'Dt-F1':>8}")
        for _, r in df.iterrows():
            print(f"{str(r.get('id',''))[:33]:<34}{r.get('MIA_paper',float('nan')):>8.3f}"
                  f"{r.get('Forget_AUC',float('nan')):>9.3f}{r.get('Forget_Macro_F1',float('nan')):>8.3f}"
                  f"{r.get('Test_AUC',float('nan')):>9.3f}{r.get('Test_Macro_F1',float('nan')):>8.3f}")
        print(chr(10) + 'Luu y: MIA cua ta dung POOL khac paper (member=retain, non-member=D_t_final 75%),'
              ' nen cot MIA chi tham khao. Df-AUC/Df-F1 la cot so sanh dang tin nhat.')
    else:
        print('chua co dong forgetmi_paper trong', RESULTS_CSV)
else:
    print('chua co', RESULTS_CSV)


In [ ]:
# Cell 5: bang ket qua day du
import os, pandas as pd
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 40)
if os.path.exists(RESULTS_CSV):
    df = pd.read_csv(RESULTS_CSV)
    cols = [c for c in ['method','checkpoint_kind','id','weight_preset','Forget_AUC','Forget_Macro_F1',
                        'Test_AUC','Test_Macro_F1','MIA','MIA_paper','forget_ce','test_ce',
                        'total_optimizer_steps','unlearn_core_hours'] if c in df.columns]
    print(f'{len(df)} dong:')
    print(df[cols].to_string(index=False))
else:
    print('chua co', RESULTS_CSV)
print(chr(10) + 'TAI VE: results_advanced.csv + perepoch_paperfix_*.csv tu tab Output.')
